# Credit Default Classification — From Scratch

This notebook trains five classifiers and computes six metrics without importing scikit-learn. NumPy is used for numerical operations and Pandas for CSV handling.

## Before running
1. Put `UCI_Credit_Card.csv` in the same folder as this notebook. Download it from the selected Kaggle dataset if needed.
2. Install dependencies if required.
3. Run the cells from top to bottom. Training may take some time because KNN and the custom decision tree are implemented directly.

The notebook saves five `.joblib` model files in `model/`, plus `model_metrics.csv` and `test_data.csv`.

In [1]:
import numpy as np
import pandas as pd
import math
import joblib
from pathlib import Path


## 1. Algorithm and metric functions
The following functions implement:
1. logistic regression,
2. a decision tree,
3. KNN,
4. Gaussian Naive Bayes,
5. linear SVM.

Six evaluation metrics:
1. Accuracy
2. AUC
3. F1
4. Precision
5. Recall
6. MCC

In [3]:
from __future__ import annotations
import math
import numpy as np


def standardize_fit(X):
    mean = X.mean(axis=0)
    scale = X.std(axis=0)
    scale[scale == 0] = 1.0
    return mean, scale


def standardize_transform(X, mean, scale):
    return (X - mean) / scale


def sigmoid(z):
    z = np.clip(z, -35, 35)
    return 1.0 / (1.0 + np.exp(-z))


def train_logistic_regression(X, y, learning_rate=0.15, epochs=700, l2=0.001):
    mean, scale = standardize_fit(X)
    Z = standardize_transform(X, mean, scale)
    n, p = Z.shape
    w = np.zeros(p, dtype=float)
    b = 0.0
    for _ in range(epochs):
        probabilities = sigmoid(Z @ w + b)
        error = probabilities - y
        grad_w = (Z.T @ error) / n + l2 * w
        grad_b = error.mean()
        w -= learning_rate * grad_w
        b -= learning_rate * grad_b
    return {"kind": "logistic", "mean": mean, "scale": scale, "weights": w, "bias": b}


def gini_impurity(y):
    if len(y) == 0:
        return 0.0
    p = float(np.mean(y))
    return 2.0 * p * (1.0 - p)


def _build_tree(X, y, depth, max_depth, min_leaf, max_thresholds):
    positives = int(y.sum())
    negatives = len(y) - positives
    node = {"prediction": int(positives >= negatives), "probability": positives / len(y)}
    if depth >= max_depth or len(y) < 2 * min_leaf or positives == 0 or negatives == 0:
        return node

    parent_gini = gini_impurity(y)
    best_gain, best_feature, best_threshold = 0.0, None, None
    n_features = X.shape[1]

    for feature in range(n_features):
        values = X[:, feature]
        # Quantile candidates keep the from-scratch tree practical on 24k rows.
        thresholds = np.unique(np.quantile(values, np.linspace(0.1, 0.9, max_thresholds)))
        for threshold in thresholds:
            left_mask = values <= threshold
            left_count = int(left_mask.sum())
            right_count = len(y) - left_count
            if left_count < min_leaf or right_count < min_leaf:
                continue
            y_left, y_right = y[left_mask], y[~left_mask]
            weighted = (left_count * gini_impurity(y_left) + right_count * gini_impurity(y_right)) / len(y)
            gain = parent_gini - weighted
            if gain > best_gain:
                best_gain, best_feature, best_threshold = gain, feature, float(threshold)

    if best_feature is None:
        return node
    mask = X[:, best_feature] <= best_threshold
    if not mask.any() or mask.all():
        return node
    node.update({
        "feature": int(best_feature),
        "threshold": best_threshold,
        "left": _build_tree(X[mask], y[mask], depth + 1, max_depth, min_leaf, max_thresholds),
        "right": _build_tree(X[~mask], y[~mask], depth + 1, max_depth, min_leaf, max_thresholds),
    })
    return node


def train_decision_tree(X, y, max_depth=8, min_samples_leaf=80, max_thresholds=9):
    tree = _build_tree(X, y, 0, max_depth, min_samples_leaf, max_thresholds)
    return {"kind": "tree", "tree": tree}


def _tree_predict_one(node, row):
    while "feature" in node:
        if row[node["feature"]] <= node["threshold"]:
            node = node["left"]
        else:
            node = node["right"]
    return node["prediction"], node["probability"]


def train_knn(X, y, k=5, max_train_rows=24000):
    mean, scale = standardize_fit(X)
    Z = standardize_transform(X, mean, scale)
    # Store all training rows by default; cap is a safeguard for larger uploaded datasets.
    if len(Z) > max_train_rows:
        rng = np.random.default_rng(42)
        chosen = np.sort(rng.choice(len(Z), max_train_rows, replace=False))
        Z, y = Z[chosen], y[chosen]
    return {"kind": "knn", "mean": mean, "scale": scale, "X_train": Z, "y_train": y.astype(int), "k": k}


def train_gaussian_nb(X, y, smoothing=1e-9):
    classes = np.array([0, 1])
    means, variances, priors = [], [], []
    for cls in classes:
        Xc = X[y == cls]
        means.append(Xc.mean(axis=0))
        variances.append(Xc.var(axis=0) + smoothing)
        priors.append(len(Xc) / len(X))
    return {"kind": "gaussian_nb", "means": np.asarray(means), "variances": np.asarray(variances), "priors": np.asarray(priors)}


def train_linear_svm(X, y, learning_rate=0.01, epochs=180, regularization=0.01):
    mean, scale = standardize_fit(X)
    Z = standardize_transform(X, mean, scale)
    targets = np.where(y == 1, 1.0, -1.0)
    n, p = Z.shape
    w = np.zeros(p, dtype=float)
    b = 0.0
    for _ in range(epochs):
        margins = targets * (Z @ w + b)
        active = margins < 1.0
        grad_w = regularization * w - (Z[active].T @ targets[active]) / n
        grad_b = -targets[active].sum() / n
        w -= learning_rate * grad_w
        b -= learning_rate * grad_b
    return {"kind": "linear_svm", "mean": mean, "scale": scale, "weights": w, "bias": b}


def predict_model(model, X):
    kind = model["kind"]
    if kind == "logistic":
        Z = standardize_transform(X, model["mean"], model["scale"])
        scores = sigmoid(Z @ model["weights"] + model["bias"])
        return (scores >= 0.5).astype(int), scores
    if kind == "tree":
        results = [_tree_predict_one(model["tree"], row) for row in X]
        predictions = np.fromiter((item[0] for item in results), dtype=int, count=len(X))
        scores = np.fromiter((item[1] for item in results), dtype=float, count=len(X))
        return predictions, scores
    if kind == "knn":
        Z = standardize_transform(X, model["mean"], model["scale"])
        train_X, train_y, k = model["X_train"], model["y_train"], model["k"]
        scores = np.empty(len(Z), dtype=float)
        # Batch queries to limit peak memory use.
        train_norm = np.sum(train_X * train_X, axis=1)
        for start in range(0, len(Z), 64):
            batch = Z[start:start + 64]
            distances = np.sum(batch * batch, axis=1)[:, None] + train_norm[None, :] - 2.0 * (batch @ train_X.T)
            nearest = np.argpartition(distances, kth=k - 1, axis=1)[:, :k]
            scores[start:start + len(batch)] = train_y[nearest].mean(axis=1)
        return (scores >= 0.5).astype(int), scores
    if kind == "gaussian_nb":
        log_likelihoods = []
        for cls in (0, 1):
            variance = model["variances"][cls]
            mean = model["means"][cls]
            log_prob = -0.5 * np.sum(np.log(2.0 * np.pi * variance) + ((X - mean) ** 2) / variance, axis=1)
            log_likelihoods.append(log_prob + np.log(model["priors"][cls]))
        log_likelihoods = np.column_stack(log_likelihoods)
        # Stable two-class softmax gives a positive-class probability for AUC.
        shifted = log_likelihoods - log_likelihoods.max(axis=1, keepdims=True)
        probabilities = np.exp(shifted)
        scores = probabilities[:, 1] / probabilities.sum(axis=1)
        return (scores >= 0.5).astype(int), scores
    if kind == "linear_svm":
        Z = standardize_transform(X, model["mean"], model["scale"])
        scores = Z @ model["weights"] + model["bias"]
        return (scores >= 0.0).astype(int), scores
    raise ValueError(f"Unknown model type: {kind}")


def confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    return tn, fp, fn, tp


def auc_score(y_true, scores):
    """ROC-AUC via the Mann-Whitney rank formula, including average ranks for ties."""
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    positives = int(np.sum(y_true == 1))
    negatives = int(np.sum(y_true == 0))
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    ranks = np.empty(len(scores), dtype=float)
    i = 0
    while i < len(scores):
        j = i + 1
        while j < len(scores) and sorted_scores[j] == sorted_scores[i]:
            j += 1
        ranks[order[i:j]] = (i + 1 + j) / 2.0
        i = j
    rank_sum_positive = ranks[y_true == 1].sum()
    return float((rank_sum_positive - positives * (positives + 1) / 2.0) / (positives * negatives))


def calculate_metrics(y_true, y_pred, scores):
    tn, fp, fn, tp = confusion_counts(y_true, y_pred)
    total = tn + fp + fn + tp
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    denominator = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn - fp * fn) / denominator) if denominator else 0.0
    return {
        "Accuracy": accuracy,
        "AUC": auc_score(y_true, scores),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "MCC": mcc,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    }


## 2. Load dataset and create a stratified train/test split

In [5]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

# The dataset CSV must be in the same folder as this notebook.
DATA_FILE = Path("UCI_Credit_Card.csv")
MODEL_DIR = Path("model")
TARGET = "default.payment.next.month"


def stratified_split(X, y, test_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    train_indices, test_indices = [], []
    for label in np.unique(y):
        indices = np.flatnonzero(y == label)
        rng.shuffle(indices)
        n_test = int(round(len(indices) * test_fraction))
        test_indices.extend(indices[:n_test])
        train_indices.extend(indices[n_test:])
    train_indices = np.asarray(train_indices)
    test_indices = np.asarray(test_indices)
    rng.shuffle(train_indices)
    rng.shuffle(test_indices)
    return X[train_indices], X[test_indices], y[train_indices], y[test_indices], test_indices

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Upload UCI_Credit_Card.csv to the same folder as this notebook, then run this cell again."
    )

df = pd.read_csv(DATA_FILE)
if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' was not found in the CSV")

y = df[TARGET].to_numpy(dtype=int)
feature_columns = [c for c in df.columns if c not in (TARGET, "ID")]
X = df[feature_columns].to_numpy(dtype=float)
X_train, X_test, y_train, y_test, test_indices = stratified_split(X, y)

print(f"Dataset shape: {df.shape}")
print(f"Features used: {len(feature_columns)}")
print(f"Training rows: {len(y_train)} | Test rows: {len(y_test)}")
print("Target distribution:", dict(zip(*np.unique(y, return_counts=True))))


Dataset shape: (30000, 25)
Features used: 23
Training rows: 24000 | Test rows: 6000
Target distribution: {np.int64(0): np.int64(23364), np.int64(1): np.int64(6636)}


## 3. Train models, calculate metrics, and save outputs

In [9]:
models = {
    "logistic_regression": train_logistic_regression(X_train, y_train),
    "decision_tree": train_decision_tree(X_train, y_train),
    "knn": train_knn(X_train, y_train, k=5),
    "naive_bayes": train_gaussian_nb(X_train, y_train),
    "support_vector_machine": train_linear_svm(X_train, y_train),
}

MODEL_DIR.mkdir(exist_ok=True)
rows = []
display_names = {
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "knn": "K-Nearest Neighbors",
    "naive_bayes": "Gaussian Naive Bayes",
    "support_vector_machine": "Linear Support Vector Machine",
}

for name, model in models.items():
    model["feature_columns"] = feature_columns
    model["display_name"] = display_names[name]
    with open(MODEL_DIR / f"{name}.pkl", "wb") as file:
        pickle.dump(model, file, protocol=pickle.HIGHEST_PROTOCOL)

    predictions, scores = predict_model(model, X_test)
    metrics = calculate_metrics(y_test, predictions, scores)
    rows.append({"Model": display_names[name], **{
        key: metrics[key] for key in ("Accuracy", "AUC", "Precision", "Recall", "F1", "MCC")
    }})
    print(display_names[name] + ": " + ", ".join(
        f"{key}={metrics[key]:.4f}" for key in ("Accuracy", "AUC", "Precision", "Recall", "F1", "MCC")
    ))

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv("model_metrics.csv", index=False)
test_df = df.iloc[test_indices].drop(columns=["ID"], errors="ignore")
test_df.to_csv("test_data.csv", index=False)

print("\nTraining complete.")
print(f"Saved {len(models)} custom models in: {MODEL_DIR.resolve()}")
print("Created model_metrics.csv and test_data.csv")
metrics_df


Logistic Regression: Accuracy=0.8095, AUC=0.7040, Precision=0.6983, Recall=0.2442, F1=0.3618, MCC=0.3328
Decision Tree: Accuracy=0.8132, AUC=0.7475, Precision=0.6435, Recall=0.3482, F1=0.4518, MCC=0.3751
K-Nearest Neighbors: Accuracy=0.7932, AUC=0.7008, Precision=0.5505, Recall=0.3534, F1=0.4305, MCC=0.3228
Gaussian Naive Bayes: Accuracy=0.6922, AUC=0.7208, Precision=0.3836, Recall=0.6458, F1=0.4813, MCC=0.3015
Linear Support Vector Machine: Accuracy=0.8050, AUC=0.6874, Precision=0.6548, Recall=0.2502, F1=0.3621, MCC=0.3174

Training complete.
Saved 5 custom models in: /home/s2026aj05071/workspace/model
Created model_metrics.csv and test_data.csv


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.809500,0.703994,0.698276,0.244160,0.361809,0.332810
1,Decision Tree,0.813167,0.747533,0.643454,0.348154,0.451834,0.375136
2,K-Nearest Neighbors,0.793167,0.700795,0.550469,0.353429,0.430473,0.322785
3,Gaussian Naive Bayes,0.692167,0.720811,0.383617,0.645818,0.481325,0.301467
4,Linear Support Vector Machine,0.805000,0.687353,0.654832,0.250188,0.362050,0.317448


## 4. Metrics overview
- **Accuracy** = (TP + TN) / (TP + TN + FP + FN)
- **Precision** = TP / (TP + FP)
- **Recall** = TP / (TP + FN)
- **F1** = 2 × Precision × Recall / (Precision + Recall)
- **MCC** = (TP×TN − FP×FN) / √((TP+FP)(TP+FN)(TN+FP)(TN+FN))
- **AUC** is calculated from prediction scores using pairwise positive/negative ranking.

These functions are custom implementations. The models are educational from-scratch implementations and may differ in performance from optimized library implementations.